# imports

for 1 home is 218 minutes so 3.5 hours

Germany 28 homes *5 
Ireland 20 homes *5
Portugal 22 homes *5 = 350

350*3.5= 1225 hours = 50 days

In [3]:
from statsmodels.tsa.stattools import adfuller, acf, pacf
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Lasso, Ridge, ElasticNet, LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import xgboost
import lightgbm
import catboost
from sktime.forecasting.arima import AutoARIMA
from sktime.forecasting.fbprophet import Prophet
from sktime.forecasting.croston import Croston
from sktime.forecasting.theta import ThetaForecaster
import optuna
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd
from pathlib import Path
import pathlib
import math
import holidays
import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
import random
import torch
import warnings
import xgboost as xgb
import lightgbm as lgb
from neuralforecast import NeuralForecast
from neuralforecast.models import (
    LSTM, GRU, PatchTST, TiDE, TCN, NBEATSx, NHITS, TFT, TSMixerx, KAN
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
GPU_DEVICE = "0"
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

# ============================================================
# BASE MODELS
# ============================================================
lasso_model = Lasso(alpha=0.01, max_iter=5000)
xgb_model = xgboost.XGBRegressor(n_jobs=-1, device="cuda", tree_method="hist", random_state=SEED)
lgb_model = lightgbm.LGBMRegressor(verbose=-1, device_type="gpu", random_state=SEED, n_jobs=-1)
cat_model = catboost.CatBoostRegressor(verbose=0, task_type="GPU", devices=GPU_DEVICE, random_seed=SEED)

rf_model = RandomForestRegressor(n_jobs=-1, random_state=SEED)
gb_model = GradientBoostingRegressor(random_state=SEED)
et_model = ExtraTreesRegressor(n_jobs=-1, random_state=SEED)
ridge_model = Ridge(alpha=1.0)
elasticnet_model = ElasticNet(alpha=0.01, l1_ratio=0.2, max_iter=5000)
linear_model = LinearRegression()
mlp_model = MLPRegressor(
    hidden_layer_sizes=(200, 100, 50),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=SEED
)
svr_model = SVR()
dt_model = DecisionTreeRegressor(random_state=SEED)
auto_arimax_model = AutoARIMA(n_jobs=-1)
prophet_model = Prophet()
croston_model = Croston()
theta_model = ThetaForecaster(deseasonalize=False)

lstm_model = LSTM(h=1)
gru_model = GRU(h=1)
patchtst_model = PatchTST(h=1,input_size=96)
tide_model = TiDE(h=1,input_size=96)
tcn_model = TCN(h=1)
nbeatsx_model = NBEATSx(h=96,input_size=96)
nhits_model = NHITS(h=1,input_size=96)
tft_model = TFT(h=1,input_size=96)
tsmixerx_model = TSMixerx(h=1,input_size=96,n_series=1)
kan_model = KAN(h=1,input_size=96)


# ============================================================
# HELPERS
# ============================================================
def make_nf_dataframe(y, X=None, unique_id="series_1"):
    df_nf = pd.DataFrame({
        "unique_id": unique_id,
        "ds": pd.to_datetime(y.index),
        "y": y.values
    })

    if X is not None:
        Xc = X.copy()
        Xc.index = pd.to_datetime(Xc.index)
        Xc = Xc.reset_index(drop=False)
        Xc = Xc.rename(columns={Xc.columns[0]: "ds"})
        df_nf = df_nf.merge(Xc, on="ds", how="left")

    return df_nf


def nf_objective_factory(nf_model_class, X_train, y_train, SEED):
    def objective(trial):
        tscv = TimeSeriesSplit(n_splits=3)
        mse_scores = []

        for train_idx, val_idx in tscv.split(X_train):
            y_train_fold = y_train.iloc[train_idx].copy()
            X_train_fold = X_train.iloc[train_idx].copy()
            y_val_fold = y_train.iloc[val_idx].copy()
            X_val_fold = X_train.iloc[val_idx].copy()

            h = len(y_val_fold)
            train_len = len(y_train_fold)

            max_input_size = min(10 * h, train_len - 1)
            if max_input_size < h:
                return float("inf")

            params = {
                "h": h,
                "input_size": trial.suggest_int("input_size", h, max(h, max_input_size)),
                "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
                "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
                "max_steps": 200,
                "scaler_type": "minmax",
                "random_seed": SEED,
            }


            if nf_model_class in [LSTM, GRU, TiDE, TCN, NBEATSx, NHITS, TFT, TSMixerx, KAN]:
                params["hist_exog_list"] = list(X_train_fold.columns)


            if nf_model_class in [LSTM, GRU]:
                params.update({
                    "encoder_n_layers": trial.suggest_int("encoder_n_layers", 1, 4),
                    "encoder_hidden_size": trial.suggest_int("encoder_hidden_size", 32, 256, step=32),
                    "encoder_dropout": trial.suggest_float("encoder_dropout", 0.0, 0.5),
                    "decoder_hidden_size": trial.suggest_int("decoder_hidden_size", 32, 256, step=32),
                    "decoder_layers": trial.suggest_int("decoder_layers", 1, 3),
                })

            elif nf_model_class == TCN:
                params.update({
                    "encoder_hidden_size": trial.suggest_int("encoder_hidden_size", 32, 256, step=32),
                    "kernel_size": trial.suggest_int("kernel_size", 2, 8),
                    "dilations": [1, 2, 4, 8],
                    "dropout": trial.suggest_float("dropout", 0.0, 0.5),
                })

            elif nf_model_class == PatchTST:
                params.update({
                    "hidden_size": trial.suggest_int("hidden_size", 16, 128, step=16),
                    "n_heads": trial.suggest_categorical("n_heads", [2, 4, 8]),
                    "patch_len": trial.suggest_categorical("patch_len", [8, 16, 24]),
                    "stride": trial.suggest_categorical("stride", [4, 8, 12]),
                    "dropout": trial.suggest_float("dropout", 0.0, 0.5),
                })

            elif nf_model_class == TiDE:
                params.update({
                    "hidden_size": trial.suggest_int("hidden_size", 32, 256, step=32),
                    "dropout": trial.suggest_float("dropout", 0.0, 0.5),
                    "num_encoder_layers": trial.suggest_int("num_encoder_layers", 1, 3),
                    "num_decoder_layers": trial.suggest_int("num_decoder_layers", 1, 3),
                })

            elif nf_model_class in [NBEATSx, NHITS]:
                params.update({
                    "mlp_units": [[trial.suggest_int("hidden_1", 64, 512, step=64),
                                   trial.suggest_int("hidden_2", 64, 512, step=64)]],
                    "n_blocks": [1, 1, 1],
                })

            elif nf_model_class == TFT:
                params.update({
                    "hidden_size": trial.suggest_int("hidden_size", 16, 128, step=16),
                    "n_head": trial.suggest_categorical("n_head", [2, 4, 8]),
                    "dropout": trial.suggest_float("dropout", 0.0, 0.5),
                })

            elif nf_model_class == TSMixerx:
                params.update({
                    "n_series": 1,
                    "n_block": trial.suggest_int("n_block", 1, 4),
                    "ff_dim": trial.suggest_int("ff_dim", 32, 256, step=32),
                    "dropout": trial.suggest_float("dropout", 0.0, 0.5),
                })

            elif nf_model_class == KAN:
                params.update({
                    "hidden_size": trial.suggest_int("hidden_size", 32, 256, step=32),
                    "dropout": trial.suggest_float("dropout", 0.0, 0.5),
                })

            try:
                if nf_model_class == PatchTST:
                    train_df_nf = make_nf_dataframe(y_train_fold, X=None, unique_id="series_1")
                else:
                    train_df_nf = make_nf_dataframe(y_train_fold, X_train_fold, unique_id="series_1")

                model = nf_model_class(**params)
                nf = NeuralForecast(models=[model], freq="15min")
                nf.fit(df=train_df_nf)

                preds = nf.predict()
                pred_col = preds.columns.difference(["unique_id", "ds"])[0]
                y_pred_fold = preds[pred_col].values

                if len(y_pred_fold) != len(y_val_fold):
                    return float("inf")

                mse_scores.append(mean_squared_error(y_val_fold.to_numpy(), y_pred_fold))

            except Exception:
                return float("inf")

        return float(np.mean(mse_scores))
    return objective



def hpo_models(X_train, y_train, models, opt_trials, forecast_horizon):
    parameters = {}

    for opt_model in models.keys():
        if opt_model == "Lasso":
            def objective_Lasso(trial):
                sugg_alpha = trial.suggest_float("alpha", 1e-5, 1)
                sugg_fit_intercept = trial.suggest_categorical("fit_intercept", [True, False])  # Include intercept or not
                sugg_selection = trial.suggest_categorical("selection", ["cyclic", "random"])  # Update strategy

                # Initialize MinMaxScaler
                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                # Scale the training data
                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                # Initialize model with suggested hyperparameters
                model = Lasso(alpha=sugg_alpha, fit_intercept=sugg_fit_intercept, selection=sugg_selection, max_iter=3000, random_state=42)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    # 🔹 Unscale predictions **before** calculating MSE
                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse = mean_squared_error(y_val_original, y_pred)  # Compute MSE in the original scale
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_Lasso = optuna.create_study(direction="minimize")
            study_Lasso.optimize(objective_Lasso, n_trials=opt_trials)
            parameters[opt_model] = study_Lasso.best_params
        

        if opt_model=="XGBoost":
            def objective_XGB(trial):

                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                }  
                model=xgboost.XGBRegressor(**params,device="cuda",tree_method="hist")
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            study_XGB=optuna.create_study(direction="minimize")
            study_XGB.optimize(objective_XGB,n_trials=opt_trials)
            parameters[opt_model]=study_XGB.best_params

        if opt_model=="LightGBM":
            def objective_LGBM(trial):

                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                } 
                model=lightgbm.LGBMRegressor(**params,verbose=-1,device_type="gpu")
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            study_LGBM=optuna.create_study(direction="minimize")
            study_LGBM.optimize(objective_LGBM,n_trials=opt_trials)
            parameters[opt_model]=study_LGBM.best_params

        if opt_model=="CatBoost":
            def objective_CatBoost(trial):

                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3,log=True),  # Log scale for better tuning
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "depth": trial.suggest_int("depth", 3, 12),  # Tree complexity
                    "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10, log=True),  # L2 regularization
                    "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),  # Randomness in sampling
                    #"colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),  # only on cpu!
                } 
                model=catboost.CatBoostRegressor(**params,task_type="GPU",devices=GPU_DEVICE, verbose=0)
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            study_CatBoost=optuna.create_study(direction="minimize")
            study_CatBoost.optimize(objective_CatBoost,n_trials=opt_trials)
            parameters[opt_model]=study_CatBoost.best_params

        if opt_model=="RandomForest":
            def objective_RF(trial):

                params = {
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),  # Number of trees
                    "max_depth": trial.suggest_int("max_depth", 3, 30),  # Tree depth
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),  # Minimum leaf size
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),  # Minimum samples per split
                    "max_features": trial.suggest_float("max_features", 0.3, 1.0),  # Feature fraction
                    "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),  # Whether to sample data with replacement
                }
                model=RandomForestRegressor(**params)
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            study_RF=optuna.create_study(direction="minimize")
            study_RF.optimize(objective_RF,n_trials=opt_trials)
            parameters[opt_model]=study_RF.best_params

        if opt_model == "GradientBoosting":
            def objective_GBR(trial):
                params = {
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
                    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                    "max_features": trial.suggest_float("max_features", 0.3, 1.0),
                }
                model = GradientBoostingRegressor(**params)
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            
            study_GBR = optuna.create_study(direction="minimize")
            study_GBR.optimize(objective_GBR, n_trials=opt_trials)
            parameters[opt_model] = study_GBR.best_params

        if opt_model == "ExtraTrees":
            def objective_ETR(trial):
                params = {
                    "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
                    "max_depth": trial.suggest_int("max_depth", 3, 30),
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
                    "max_features": trial.suggest_float("max_features", 0.3, 1.0),
                    "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
                }
                model = ExtraTreesRegressor(**params)
                
                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)
                    
                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)
            
            study_ETR = optuna.create_study(direction="minimize")
            study_ETR.optimize(objective_ETR, n_trials=opt_trials)
            parameters[opt_model] = study_ETR.best_params

        if opt_model == "Ridge":
            def objective_Ridge(trial):
                params = {
                    "alpha": trial.suggest_float("alpha", 1e-5, 10, log=True),
                    "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False])
                }

                # Initialize MinMaxScaler
                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                # Scale the training data
                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                # Initialize model with suggested hyperparameters
                model = Ridge(**params)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    # 🔹 Unscale predictions **before** calculating MSE
                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse = mean_squared_error(y_val_original, y_pred)  # Compute MSE in the original scale
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_Ridge = optuna.create_study(direction="minimize")
            study_Ridge.optimize(objective_Ridge, n_trials=opt_trials)
            parameters[opt_model] = study_Ridge.best_params

        if opt_model == "ElasticNet":
            def objective_ElasticNet(trial):
                params = {
                    "alpha": trial.suggest_float("alpha", 1e-5, 10.0, log=True),  # Regularization strength
                    "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),  # Balance between L1 (Lasso) and L2 (Ridge)
                    "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),  # Whether to fit the intercept
                }

                # Initialize MinMaxScaler
                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                # Scale the training data
                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                # Initialize ElasticNet model with suggested hyperparameters
                model = ElasticNet(**params)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    # 🔹 Unscale predictions and validation target before calculating MSE
                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse = mean_squared_error(y_val_original, y_pred)  # Compute MSE in the original scale
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_ElasticNet = optuna.create_study(direction="minimize")
            study_ElasticNet.optimize(objective_ElasticNet, n_trials=opt_trials)
            parameters[opt_model] = study_ElasticNet.best_params


        if opt_model == "SVR":
            def objective_SVR(trial):
                params = {
                    "C": trial.suggest_float("C", 1e-3, 100.0, log=True),  # Regularization parameter
                    "epsilon": trial.suggest_float("epsilon", 1e-4, 1.0, log=True),  # Epsilon-tube within which no penalty is given
                    "kernel": trial.suggest_categorical("kernel", ["linear", "poly", "rbf", "sigmoid"]),  # Kernel type
                    "gamma": trial.suggest_categorical("gamma", ["scale", "auto"]),  # Kernel coefficient
                    "shrinking": trial.suggest_categorical("shrinking", [True, False]),  # Shrinking heuristic
                }

                # Handle "degree" only if kernel is "poly"
                if params["kernel"] == "poly":
                    params["degree"] = trial.suggest_int("degree", 2, 5)  # Polynomial kernel degree

                # Initialize MinMaxScaler
                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                # Scale the training data
                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                # Initialize SVR model with suggested hyperparameters
                model = SVR(**params)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    # 🔹 Unscale predictions and validation target before calculating MSE
                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse = mean_squared_error(y_val_original, y_pred)  # Compute MSE in the original scale
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_SVR = optuna.create_study(direction="minimize")
            study_SVR.optimize(objective_SVR, n_trials=opt_trials)
            parameters[opt_model] = study_SVR.best_params


        if opt_model == "DecisionTree":
            def objective_DTR(trial):
                params = {
                    "max_depth": trial.suggest_int("max_depth", 3, 30),  # Maximum depth of the tree
                    "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),  # Minimum samples to split
                    "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),  # Minimum samples per leaf
                    "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),  # Feature selection
                    "splitter": trial.suggest_categorical("splitter", ["best", "random"]),  # Splitting strategy
                }

                model = DecisionTreeRegressor(**params)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred = model.predict(X_val_fold)

                    mse = mean_squared_error(y_val_fold, y_pred)  
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_DTR = optuna.create_study(direction="minimize")
            study_DTR.optimize(objective_DTR, n_trials=opt_trials)
            parameters[opt_model] = study_DTR.best_params


        if opt_model == "MLP":
            def objective_MLP(trial):
                # Determine the number of layers (2 to 4)
                num_layers = trial.suggest_int("num_layers", 2, 4)

                # Define hidden layer sizes with different neuron counts per layer
                hidden_layer_sizes = tuple(
                    trial.suggest_int(f"layer_{i+1}", 50, 500, step=50) for i in range(num_layers)
                )

                params = {
                    "hidden_layer_sizes": hidden_layer_sizes,  # Variable hidden layer structure
                    "activation": trial.suggest_categorical("activation", ["identity", "logistic", "tanh", "relu"]),  # Activation function
                    "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),  # L2 regularization term
                    "learning_rate": trial.suggest_categorical("learning_rate", ["constant", "invscaling", "adaptive"]),  # Learning rate strategy
                    "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-1, log=True),  # Initial learning rate
                    "batch_size": trial.suggest_categorical("batch_size", ["auto", 32, 64, 128]),  # Batch size
                }

                # Initialize MinMaxScaler
                X_scaler = MinMaxScaler()
                y_scaler = MinMaxScaler()

                # Scale the training data
                X_train_scaled = X_scaler.fit_transform(X_train)
                y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).ravel()

                # Initialize MLP model with suggested hyperparameters
                model = MLPRegressor(**params, max_iter=300)

                # Time Series Cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                for train_idx, val_idx in tscv.split(X_train_scaled):
                    X_train_fold, X_val_fold = X_train_scaled[train_idx], X_train_scaled[val_idx]
                    y_train_fold, y_val_fold = y_train_scaled[train_idx], y_train_scaled[val_idx]

                    model.fit(X_train_fold, y_train_fold)
                    y_pred_scaled = model.predict(X_val_fold)

                    # 🔹 Unscale predictions and validation target before calculating MSE
                    y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
                    y_val_original = y_scaler.inverse_transform(y_val_fold.reshape(-1, 1)).ravel()

                    mse = mean_squared_error(y_val_original, y_pred)  # Compute MSE in the original scale
                    mse_scores.append(mse)

                return np.mean(mse_scores)

            study_MLP = optuna.create_study(direction="minimize")
            study_MLP.optimize(objective_MLP, n_trials=opt_trials)

            best_params = study_MLP.best_params
            num_layers = best_params.pop("num_layers")

            # Extract hidden layer sizes from the best params
            hidden_layer_sizes = tuple(best_params.pop(f"layer_{i+1}") for i in range(num_layers))
            best_params["hidden_layer_sizes"] = hidden_layer_sizes  # Store correctly

            parameters[opt_model] = best_params  # Store the final result properly


        if opt_model == "Prophet":
            def objective_prophet_ext(trial):
                """
                Objective function for tuning Prophet hyperparameters with Optuna,
                including exogenous features (X).
                """

                # 1) Suggest Prophet hyperparameters via Optuna
                params = {
                    "seasonality_mode": trial.suggest_categorical("seasonality_mode", ["additive", "multiplicative"]),
                    "seasonality_prior_scale": trial.suggest_float("seasonality_prior_scale", 0.01, 10.0),
                    "changepoint_prior_scale": trial.suggest_float("changepoint_prior_scale", 0.001, 0.5),
                    "holidays_prior_scale": trial.suggest_float("holidays_prior_scale", 0.01, 10.0),
                }

                # 2) Create a time series splitter
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                # 3) Perform the time series CV
                for train_idx, val_idx in tscv.split(X_train):
                    y_train_fold = y_train.iloc[train_idx]
                    X_train_fold = X_train.iloc[train_idx]  # exogenous features
                    y_val_fold = y_train.iloc[val_idx]
                    X_val_fold = X_train.iloc[val_idx]      # exogenous features

                    # Instantiate Prophet with the hyperparameters
                    model = Prophet(**params)

                    model.fit(y_train_fold, X=X_train_fold)

                    # 5) Predict on the validation fold
                    # We create an fh that covers the length of the validation fold
                    fh_val = np.arange(1, len(y_val_fold) + 1)

                    # Prophet also takes exogenous data for the forecast step
                    y_pred_fold = model.predict(fh=fh_val, X=X_val_fold)

                    # 6) Compute MSE for this fold
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred_fold))

                # 7) Return average MSE as the objective to minimize
                return np.mean(mse_scores)

            # Create and run the study
            study_prophet_ext = optuna.create_study(direction="minimize")
            study_prophet_ext.optimize(objective_prophet_ext, n_trials=opt_trials)  # example: 20 trials

            # Extract the best hyperparameters
            parameters[opt_model] = study_prophet_ext.best_params

        if opt_model == "Croston":
            def objective_croston(trial):
                """
                Objective function for tuning Croston hyperparameters with Optuna.
                """

                # 1) Suggest hyperparameters
                params = {
                    "smoothing": trial.suggest_float("smoothing", 0.01, 1.0)  # Smoothing factor (0.01 to 1)
                }

                # 2) Time series cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                # 3) Perform time series CV
                for train_idx, val_idx in tscv.split(y_train):
                    y_train_fold = y_train.iloc[train_idx]
                    y_val_fold = y_train.iloc[val_idx]

                    # 4) Train Croston model
                    model = Croston(**params)
                    model.fit(y_train_fold)

                    # 5) Predict
                    fh_val = np.arange(1, len(y_val_fold) + 1)
                    y_pred_fold = model.predict(fh=fh_val)

                    # 6) Compute MSE for this fold
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred_fold))

                # 7) Return average MSE
                return np.mean(mse_scores)

            # 8) Run Optuna study
            study_croston = optuna.create_study(direction="minimize")
            study_croston.optimize(objective_croston, n_trials=opt_trials)

            # 9) Store best parameters
            parameters[opt_model] = study_croston.best_params

        if opt_model == "Theta":
            def objective_theta(trial):
                """
                Objective function for tuning ThetaForecaster hyperparameters with Optuna.
                """
                if (y_train < 0).any() or (y_train == 0).any():
                    deseasonalize_option = False  # Force additive seasonality
                else:
                    deseasonalize_option = trial.suggest_categorical("deseasonalize", [True, False])
                # 1) Suggest hyperparameters
                params = {
                    "initial_level": trial.suggest_float("initial_level", 0.01, 1.0),  # SES smoothing factor
                    "deseasonalize": deseasonalize_option,
                    "sp": trial.suggest_int("sp", 1, min(36, len(y_train)//2 ))  # 96 for daily seasonality in 15-min data
                }

                # 2) Time series cross-validation
                tscv = TimeSeriesSplit(n_splits=3)
                mse_scores = []

                # 3) Perform time series CV
                for train_idx, val_idx in tscv.split(y_train):
                    y_train_fold = y_train.iloc[train_idx]
                    y_val_fold = y_train.iloc[val_idx]

                    # 🔹 Ensure frequency is set
                    if y_train_fold.index.freq is None:
                        y_train_fold.index.freq = pd.infer_freq(y_train_fold.index)

                    # 4) Train ThetaForecaster
                    model = ThetaForecaster(**params)
                    model.fit(y_train_fold)

                    # 5) Predict
                    fh_val = np.arange(1, len(y_val_fold) + 1)
                    y_pred_fold = model.predict(fh=fh_val)

                    # 6) Compute MSE for this fold
                    mse_scores.append(mean_squared_error(y_val_fold, y_pred_fold))

                # 7) Return average MSE
                return np.mean(mse_scores)

            # 8) Run Optuna study
            study_theta = optuna.create_study(direction="minimize")
            study_theta.optimize(objective_theta, n_trials=opt_trials)

            # 9) Store best parameters
            parameters[opt_model] = study_theta.best_params

        if opt_model == "LSTM":
            study = optuna.create_study(direction="minimize")
            study.optimize(nf_objective_factory(LSTM, X_train, y_train, SEED), n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        if opt_model == "GRU":
            study = optuna.create_study(direction="minimize")
            study.optimize(nf_objective_factory(GRU, X_train, y_train, SEED), n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        if opt_model == "PatchTST":
            study = optuna.create_study(direction="minimize")
            study.optimize(nf_objective_factory(PatchTST, X_train, y_train, SEED), n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        if opt_model == "TiDE":
            study = optuna.create_study(direction="minimize")
            study.optimize(nf_objective_factory(TiDE, X_train, y_train, SEED), n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        if opt_model == "TCN":
            study = optuna.create_study(direction="minimize")
            study.optimize(nf_objective_factory(TCN, X_train, y_train, SEED), n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        if opt_model == "NBEATSx":
            study = optuna.create_study(direction="minimize")
            study.optimize(nf_objective_factory(NBEATSx, X_train, y_train, SEED), n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        if opt_model == "NHITS":
            study = optuna.create_study(direction="minimize")
            study.optimize(nf_objective_factory(NHITS, X_train, y_train, SEED), n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        if opt_model == "TFT":
            study = optuna.create_study(direction="minimize")
            study.optimize(nf_objective_factory(TFT, X_train, y_train, SEED), n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        if opt_model == "TSMixerx":
            study = optuna.create_study(direction="minimize")
            study.optimize(nf_objective_factory(TSMixerx, X_train, y_train, SEED), n_trials=opt_trials)
            parameters[opt_model] = study.best_params

        if opt_model == "KAN":
            study = optuna.create_study(direction="minimize")
            study.optimize(nf_objective_factory(KAN, X_train, y_train, SEED), n_trials=opt_trials)
            parameters[opt_model] = study.best_params



    return parameters


def forecasting_single_home(
    project_path,
    df_full,
    target_col,
    date,
    forecast_end_date,
    forecast_horizon,
    training_size,
    feature_selection,
    plot_forecast,
    hyperparameter_opt,
    models,
    country,
    day_name,
    opt_trials=30
):
    """
    Forecast a single home series using weather already inside df_full.
    """

    df_full = df_full.copy()
    df_full.index = pd.to_datetime(df_full.index)

    weather_feature_list = [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "precipitation",
        "direct_radiation"
    ]

    df_original = df_full[[target_col]].copy()
    df_weather = df_full[weather_feature_list].copy()

    target = target_col

    initial_date = pd.Timestamp(date)
    end_date = pd.Timestamp(forecast_end_date)

    time_step = int((df_original.index[1] - df_original.index[0]).total_seconds() / 60)
    step_size = pd.Timedelta(minutes=forecast_horizon * time_step)
    n_iterations = math.ceil((end_date - initial_date) / step_size)

    all_predictions = []

    for i in range(n_iterations):
        print(f"\nHome: {target_col} | iteration {i+1}/{n_iterations}")

        current_date = initial_date + i * step_size

        df_target_hist = df_original[df_original.index < current_date].copy()
        df_target_hist = df_target_hist.tail(training_size)

        start_timestamp = df_target_hist.index.min()
        end_timestamp = df_weather.index.max()

        df = pd.merge(df_target_hist, df_weather, left_index=True, right_index=True, how="outer")
        df = df.loc[start_timestamp:end_timestamp]

        first_nan_index = df[df[target].isna()].index.min()

        if pd.isna(first_nan_index):
            filtered_df = pd.DataFrame()
        else:
            start_position = df.index.get_loc(first_nan_index)
            end_position = min(start_position + forecast_horizon, len(df))
            filtered_df = df.iloc[:end_position].copy()

        # -------------------------
        # Feature engineering
        # -------------------------
        for lag in range(forecast_horizon, forecast_horizon * 2 + 1):
            filtered_df[f"{target}_lag_{lag}"] = df[target].shift(lag)

        for lag in range(1, forecast_horizon + 1):
            for g in weather_feature_list:
                filtered_df[f"{g}_lag_{lag}"] = df[g].shift(lag)

        filtered_df.index = pd.to_datetime(filtered_df.index)

        filtered_df["minute"] = filtered_df.index.minute
        filtered_df["hour"] = filtered_df.index.hour
        filtered_df["day_of_week"] = filtered_df.index.dayofweek
        filtered_df["day_of_year"] = filtered_df.index.dayofyear
        filtered_df["week"] = filtered_df.index.isocalendar().week.astype(int)
        filtered_df["month"] = filtered_df.index.month
        filtered_df["year"] = filtered_df.index.year
        filtered_df["is_weekend"] = (filtered_df["day_of_week"] >= 5).astype(int)

        if country == "Germany":
            holiday_calendar = holidays.Germany()
        elif country == "Ireland":
            holiday_calendar = holidays.Ireland()
        elif country == "Portugal":
            holiday_calendar = holidays.Portugal()
        else:
            holiday_calendar = holidays.Germany()

        filtered_df["holiday"] = filtered_df.index.to_series().apply(lambda x: 1 if x in holiday_calendar else 0)

        minute_period = 60
        hour_period = 24
        week_period = 7
        month_period = 12
        year_period = 365.25

        filtered_df["minute_sin"] = np.sin(2 * np.pi * filtered_df["minute"] / minute_period)
        filtered_df["minute_cos"] = np.cos(2 * np.pi * filtered_df["minute"] / minute_period)
        filtered_df["hour_sin"] = np.sin(2 * np.pi * filtered_df["hour"] / hour_period)
        filtered_df["hour_cos"] = np.cos(2 * np.pi * filtered_df["hour"] / hour_period)
        filtered_df["dayofweek_sin"] = np.sin(2 * np.pi * filtered_df["day_of_week"] / week_period)
        filtered_df["dayofweek_cos"] = np.cos(2 * np.pi * filtered_df["day_of_week"] / week_period)
        filtered_df["dayofyear_sin"] = np.sin(2 * np.pi * filtered_df["day_of_year"] / year_period)
        filtered_df["dayofyear_cos"] = np.cos(2 * np.pi * filtered_df["day_of_year"] / year_period)
        filtered_df["week_sin"] = np.sin(2 * np.pi * filtered_df["week"] / week_period)
        filtered_df["week_cos"] = np.cos(2 * np.pi * filtered_df["week"] / week_period)
        filtered_df["month_sin"] = np.sin(2 * np.pi * filtered_df["month"] / month_period)
        filtered_df["month_cos"] = np.cos(2 * np.pi * filtered_df["month"] / month_period)

        filtered_df = filtered_df.dropna(subset=[col for col in filtered_df.columns if col != target])

        split_point = len(filtered_df) - forecast_horizon
        if split_point <= 0:
            raise ValueError(f"Not enough data for {target_col}.")

        y_train = filtered_df[target].iloc[:split_point]
        y_test = filtered_df[target].iloc[split_point:]

        X_train = filtered_df.drop(columns=[target]).iloc[:split_point]
        X_test = filtered_df.drop(columns=[target]).iloc[split_point:]

        # -------------------------
        # Feature selection
        # -------------------------
        if feature_selection:
            correlations = X_train.corrwith(y_train)
            correlations = correlations.replace([np.inf, -np.inf], np.nan).dropna()

            if len(correlations) == 0:
                raise ValueError(f"No valid correlations could be computed for {target}.")

            selected_features = correlations[abs(correlations) >= 0.1].index.tolist()

            if len(selected_features) == 0:
                top_k = min(10, len(correlations))
                selected_features = correlations.abs().sort_values(ascending=False).head(top_k).index.tolist()
                print(f"Warning: no features passed threshold for {target}. Using top {top_k} correlated features instead.")

            X_train = X_train[selected_features]
            X_test = X_test[selected_features]

        print(f"{target} | X_train shape after feature selection: {X_train.shape}")


        # -------------------------
        # HPO
        # -------------------------
        if hyperparameter_opt:
            opt_parameters = hpo_models(X_train, y_train, models, opt_trials, forecast_horizon)
        else:
            opt_parameters = {
                "Lasso": {
                    "alpha": 0.01,
                    "fit_intercept": True,
                    "selection": "cyclic"
                },
                "Ridge": {
                    "alpha": 1.0,
                    "fit_intercept": True
                },
                "ElasticNet": {
                    "alpha": 0.01,
                    "l1_ratio": 0.5,
                    "fit_intercept": True
                },
                "LinearRegression": {},
                "SVR": {
                    "C": 1.0,
                    "epsilon": 0.1,
                    "kernel": "rbf",
                    "gamma": "scale"
                },
                "DecisionTree": {
                    "max_depth": 10,
                    "min_samples_split": 2,
                    "min_samples_leaf": 1,
                    "max_features": None,
                    "splitter": "best"
                },
                "RandomForest": {
                    "n_estimators": 200,
                    "max_depth": 15,
                    "min_samples_split": 2,
                    "min_samples_leaf": 1,
                    "max_features": 0.7,
                    "bootstrap": True
                },
                "GradientBoosting": {
                    "learning_rate": 0.05,
                    "n_estimators": 200,
                    "max_depth": 3,
                    "min_samples_split": 2,
                    "min_samples_leaf": 1,
                    "subsample": 1.0,
                    "max_features": 0.7
                },
                "ExtraTrees": {
                    "n_estimators": 200,
                    "max_depth": 15,
                    "min_samples_split": 2,
                    "min_samples_leaf": 1,
                    "max_features": 0.7,
                    "bootstrap": False
                },
                "MLP": {
                    "hidden_layer_sizes": (200, 100),
                    "activation": "relu",
                    "alpha": 1e-4,
                    "learning_rate": "adaptive",
                    "learning_rate_init": 1e-3,
                    "batch_size": 64
                },
                "XGBoost": {
                    "learning_rate": 0.05,
                    "n_estimators": 200,
                    "max_depth": 6,
                    "min_child_weight": 1,
                    "subsample": 0.8,
                    "colsample_bytree": 0.8
                },
                "LightGBM": {
                    "learning_rate": 0.05,
                    "n_estimators": 200,
                    "max_depth": 6,
                    "min_child_weight": 1,
                    "subsample": 0.8,
                    "colsample_bytree": 0.8
                },
                "CatBoost": {
                    "learning_rate": 0.05,
                    "n_estimators": 200,
                    "depth": 6,
                    "l2_leaf_reg": 3.0,
                    "bagging_temperature": 0.0
                },
                "AutoARIMAX": {},
                "Prophet": {
                    "seasonality_mode": "additive",
                    "seasonality_prior_scale": 10.0,
                    "changepoint_prior_scale": 0.05,
                    "holidays_prior_scale": 10.0
                },
                "Croston": {
                    "smoothing": 0.1
                },
                "Theta": {
                    "initial_level": 0.2,
                    "deseasonalize": False,
                    "sp": min(96, max(2, len(y_train) // 10))
                },
                "LSTM": {
                    "input_size": forecast_horizon,
                    "encoder_n_layers": 2,
                    "encoder_hidden_size": 128,
                    "encoder_dropout": 0.0,
                    "decoder_hidden_size": 128,
                    "decoder_layers": 1,
                    "learning_rate": 1e-3,
                    "batch_size": 32
                },
                "GRU": {
                    "input_size": forecast_horizon,
                    "encoder_n_layers": 2,
                    "encoder_hidden_size": 128,
                    "encoder_dropout": 0.0,
                    "decoder_hidden_size": 128,
                    "decoder_layers": 1,
                    "learning_rate": 1e-3,
                    "batch_size": 32
                },
                "PatchTST": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "hidden_size": 64,
                    "n_heads": 4,
                    "patch_len": 16,
                    "stride": 8,
                    "dropout": 0.0
                },
                "TiDE": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "hidden_size": 128,
                    "dropout": 0.0,
                    "num_encoder_layers": 2,
                    "num_decoder_layers": 2
                },
                "TCN": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "encoder_hidden_size": 128,
                    "kernel_size": 3,
                    "dropout": 0.0
                },
                "NBEATSx": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "mlp_units": [[256, 256]],
                    "n_blocks": [1, 1, 1]
                },
                "NHITS": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "mlp_units": [[256, 256]],
                    "n_blocks": [1, 1, 1]
                },
                "TFT": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "hidden_size": 64,
                    "n_head": 4,
                    "dropout": 0.0
                },
                "TSMixerx": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "n_series": 1,
                    "n_block": 2,
                    "ff_dim": 64,
                    "dropout": 0.0
                },
                "KAN": {
                    "input_size": forecast_horizon,
                    "learning_rate": 1e-3,
                    "batch_size": 32,
                    "hidden_size": 128,
                    "dropout": 0.0
                }
            }


        # -------------------------
        # Reinitialize models
        # -------------------------
        reinitialized_models = {}

        for name, params in opt_parameters.items():
            if name in models:
                if name == "Lasso":
                    reinitialized_models[name] = Lasso(**params, max_iter=5000)

                elif name == "XGBoost":
                    reinitialized_models[name] = xgboost.XGBRegressor(
                        **params,
                        n_jobs=-1,
                        random_state=SEED,
                        device="cuda",
                        tree_method="hist"
                    )

                elif name == "LightGBM":
                    reinitialized_models[name] = lightgbm.LGBMRegressor(
                        **params,
                        n_jobs=-1,
                        verbose=-1,
                        random_state=SEED,
                        device_type="gpu"
                    )

                elif name == "CatBoost":
                    reinitialized_models[name] = catboost.CatBoostRegressor(
                        **params,
                        verbose=0,
                        random_seed=SEED,
                        task_type="GPU",
                        devices=GPU_DEVICE
                    )
                elif name == "RandomForest":
                    reinitialized_models[name] = RandomForestRegressor(**params, n_jobs=-1, random_state=SEED)

                elif name == "GradientBoosting":
                    reinitialized_models[name] = GradientBoostingRegressor(**params, random_state=SEED)

                elif name == "ExtraTrees":
                    reinitialized_models[name] = ExtraTreesRegressor(**params, n_jobs=-1, random_state=SEED)

                elif name == "Ridge":
                    reinitialized_models[name] = Ridge(**params)

                elif name == "ElasticNet":
                    reinitialized_models[name] = ElasticNet(**params, max_iter=5000)

                elif name == "LinearRegression":
                    reinitialized_models[name] = LinearRegression()

                elif name == "MLP":
                    reinitialized_models[name] = MLPRegressor(**params, max_iter=1000, random_state=SEED)

                elif name == "SVR":
                    reinitialized_models[name] = SVR(**params)

                elif name == "DecisionTree":
                    reinitialized_models[name] = DecisionTreeRegressor(**params, random_state=SEED)

                elif name == "AutoARIMAX":
                    reinitialized_models[name] = AutoARIMA(sp=96, suppress_warnings=True)

                elif name == "Prophet":
                    reinitialized_models[name] = Prophet(**params)

                elif name == "Croston":
                    reinitialized_models[name] = Croston(**params)

                elif name == "Theta":
                    reinitialized_models[name] = ThetaForecaster(**params)


                elif name == "PatchTST":
                    reinitialized_models[name] = PatchTST(
                        h=forecast_horizon,
                        input_size=params.get("input_size", forecast_horizon),
                        learning_rate=params.get("learning_rate", 1e-3),
                        batch_size=params.get("batch_size", 32),
                        hidden_size=params.get("hidden_size", 128),
                        n_heads=params.get("n_heads", 4),
                        patch_len=params.get("patch_len", 16),
                        stride=params.get("stride", 8),
                        dropout=params.get("dropout", 0.0),
                        max_steps=200,
                        scaler_type="minmax",
                        random_seed=SEED
                    )

                elif name == "TSMixerx":
                    reinitialized_models[name] = TSMixerx(
                        h=forecast_horizon,
                        input_size=params.get("input_size", forecast_horizon),
                        n_series=1,
                        learning_rate=params.get("learning_rate", 1e-3),
                        batch_size=params.get("batch_size", 32),
                        n_block=params.get("n_block", 2),
                        ff_dim=params.get("ff_dim", 64),
                        dropout=params.get("dropout", 0.0),
                        max_steps=200,
                        scaler_type="minmax",
                        random_seed=SEED,
                        hist_exog_list=list(X_train.columns)
                    )


                elif name in ["LSTM", "GRU", "TiDE", "TCN", "NBEATSx", "NHITS", "TFT", "KAN"]:
                    nf_class_map = {
                        "LSTM": LSTM,
                        "GRU": GRU,
                        "TiDE": TiDE,
                        "TCN": TCN,
                        "NBEATSx": NBEATSx,
                        "NHITS": NHITS,
                        "TFT": TFT,
                        "KAN": KAN,
                    }

                    base_kwargs = {
                        "h": forecast_horizon,
                        "input_size": params.get("input_size", forecast_horizon),
                        "learning_rate": params.get("learning_rate", 1e-3),
                        "batch_size": params.get("batch_size", 32),
                        "max_steps": 200,
                        "scaler_type": "minmax",
                        "random_seed": SEED,
                        "hist_exog_list": list(X_train.columns),
                    }

                    extra_kwargs = {k: v for k, v in params.items() if k not in base_kwargs}
                    reinitialized_models[name] = nf_class_map[name](**base_kwargs, **extra_kwargs)

        models_current = reinitialized_models

        # -------------------------
        # Scaling
        # -------------------------
        scaler = MinMaxScaler()
        scaled_target = MinMaxScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        y_train_scaled = scaled_target.fit_transform(y_train.values.reshape(-1, 1)).flatten()

        y_preds = {}

        # -------------------------
        # Train
        # -------------------------
        for name, model in models_current.items():
            
            print(f"\n{'='*60}")
            print(f"MODEL: {name}")
            print(f"HOME: {target_col}")
            print(f"{'='*60}")
            print("Starting training...")

            if name in ["Lasso", "Ridge", "ElasticNet", "LinearRegression", "MLP", "SVR"]:
                model.fit(X_train_scaled, y_train_scaled)

            elif name in ["AutoARIMAX", "Prophet"]:
                model.fit(y_train, X=X_train)

            elif name in ["Croston", "Theta"]:
                if y_train.index.freq is None:
                    inferred_freq = pd.infer_freq(y_train.index)
                    if inferred_freq is None:
                        raise ValueError(f"Cannot infer frequency for {target_col}.")
                    y_train = y_train.asfreq(inferred_freq)
                model.fit(y_train)

            elif name == "PatchTST":
                train_df_nf = make_nf_dataframe(y_train, X=None, unique_id="series_1")
                nf_model = NeuralForecast(models=[model], freq="15min")
                nf_model.fit(df=train_df_nf)
                models_current[name] = nf_model



            elif name == "TSMixerx":
                train_df_nf = make_nf_dataframe(y_train, X_train, unique_id="series_1")
                nf_model = NeuralForecast(models=[model], freq="15min")
                nf_model.fit(df=train_df_nf)
                models_current[name] = nf_model

            elif name in ["LSTM", "GRU","TiDE", "TCN", "NBEATSx", "NHITS", "TFT", "KAN"]:
                train_df_nf = make_nf_dataframe(y_train, X_train, unique_id="series_1")
                nf_model = NeuralForecast(models=[model], freq="15min")
                nf_model.fit(df=train_df_nf)
                models_current[name] = nf_model

            else:
                model.fit(X_train, y_train)

            print(f"{name} training complete.")

        # -------------------------
        # Predict
        # -------------------------
        for name, model in models_current.items():
            if name in ["Lasso", "Ridge", "ElasticNet", "LinearRegression", "MLP", "SVR"]:
                y_preds[name] = scaled_target.inverse_transform(
                    model.predict(X_test_scaled).reshape(-1, 1)
                ).flatten()

            elif name in ["AutoARIMAX", "Prophet"]:
                fh = np.arange(1, len(y_test) + 1)
                y_pred_auto = model.predict(fh=fh, X=X_test)
                y_pred_auto.index = y_test.index
                y_preds[name] = y_pred_auto.values

            elif name in ["Croston", "Theta"]:
                fh = np.arange(1, len(y_test) + 1)
                y_pred_uni = model.predict(fh=fh)
                y_pred_uni.index = y_test.index
                y_preds[name] = y_pred_uni.values

            elif name in ["LSTM", "GRU", "PatchTST", "TiDE", "TCN", "NBEATSx", "NHITS", "TFT", "TSMixerx", "KAN"]:
                preds_nf = model.predict()
                pred_col = preds_nf.columns.difference(["unique_id", "ds"])[0]
                preds_nf = preds_nf.copy()
                preds_nf["ds"] = pd.to_datetime(preds_nf["ds"])
                preds_nf = preds_nf.set_index("ds")
                y_pred_nf = preds_nf.loc[y_test.index, pred_col]
                y_preds[name] = y_pred_nf.values

            else:
                y_preds[name] = model.predict(X_test)

        y_pred_df = pd.DataFrame(y_preds, index=y_test.index)
        y_actual = df_original.loc[y_test.index, target]

        inferred_freq = pd.infer_freq(df_original.index)
        if inferred_freq:
            freq_timedelta = pd.to_timedelta(inferred_freq)
            freq_minutes = freq_timedelta.total_seconds() / 60
        else:
            raise ValueError("Could not infer frequency from the dataset index.")

        time_offset = pd.Timedelta(minutes=freq_minutes * forecast_horizon)
        y_pred_naive = df_original.loc[y_test.index - time_offset, target]

        mse_scores = {name: mean_squared_error(y_actual, y_preds[name]) for name in y_preds.keys()}
        
        mse_scores["Naive Forecast"] = mean_squared_error(y_actual, y_pred_naive)

        for name, mse in mse_scores.items():
            print(f"MSE - {target_col} - {name}: {mse:.5f}")

        y_pred_df["Naive Forecast"] = y_pred_naive.values
        y_pred_df["Actual"] = y_actual.values
        all_predictions.append(y_pred_df)

    # -------------------------
    # Save outputs
    # -------------------------
    final_predictions = pd.concat(all_predictions)
    final_predictions.index.name = "timestamp"

    out_dir = pathlib.Path(project_path) / "Outputs"
    out_dir.mkdir(parents=True, exist_ok=True)

    pred_file = out_dir / f"final_predictions_{country}_{day_name}_{target_col}_{forecast_horizon}_steps.csv"
    final_predictions.to_csv(pred_file, index=True)

    print(f"CSV saved at: {pred_file}")

    param_file = out_dir / f"opt_parameters_{country}_{day_name}_{target_col}_{forecast_horizon}_steps.json"
    with open(param_file, "w") as f:
        json.dump(opt_parameters, f, indent=4)

    print(f"JSON saved at: {param_file}")

    if plot_forecast:
        plot_dir = pathlib.Path(out_dir) / "data" / "plots" / country / day_name
        plot_dir.mkdir(parents=True, exist_ok=True)

        plot_file = plot_dir / f"actual_vs_predictions_{country}_{day_name}_{target_col}_{forecast_horizon}_steps.png"

        plt.figure(figsize=(12, 6))
        colors = list(mcolors.TABLEAU_COLORS.values())

        first_timestamp = final_predictions.index.min()
        previous_day_start = first_timestamp - pd.Timedelta(days=1)

        previous_day_actual = df_original.loc[previous_day_start:first_timestamp, target]
        current_actual = final_predictions["Actual"]
        actual_combined = pd.concat([previous_day_actual, current_actual])

        plt.plot(actual_combined.index, actual_combined, label="Actual", color="black", linewidth=2)

        for i, column in enumerate(final_predictions.columns):
            if column != "Actual":
                plt.plot(final_predictions.index, final_predictions[column], label=column, color=colors[i % len(colors)])

        plt.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="black")
        plt.xlabel("Time")
        plt.ylabel("Value")
        plt.title(f"Forecast vs Actual - {target_col}")
        plt.savefig(plot_file, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"Plot saved at: {plot_file}")



Seed set to 1
Seed set to 1
Seed set to 1
Seed set to 1
Seed set to 1
Seed set to 1
Seed set to 1
Seed set to 1
Seed set to 1
Seed set to 1


# Run

In [ ]:
# ============================================================
# MAIN RUN - ALL COUNTRIES / ALL DAYS / ALL HOMES
# ============================================================

models = {
    "Lasso": lasso_model,
    "XGBoost": xgb_model,
    "LightGBM": lgb_model,
    "CatBoost": cat_model,
    "RandomForest": rf_model,
    "GradientBoosting": gb_model,
    "ExtraTrees": et_model,
    "Ridge": ridge_model,
    "ElasticNet": elasticnet_model,
    "LinearRegression": linear_model,
    "MLP": mlp_model,
    "SVR": svr_model,
    "DecisionTree": dt_model,
    "LSTM": lstm_model,
    "GRU": gru_model,
    "TCN": tcn_model,
    "PatchTST": patchtst_model,
    "TiDE": tide_model,
    "NBEATSx": nbeatsx_model,
    "NHITS": nhits_model,
#    "TFT": tft_model,
    "TSMixerx": tsmixerx_model,
    "KAN": kan_model
    }

project_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone"
days_json_path = pathlib.Path(project_path) / "dataset_days.json"

#countries = ["Germany", "Ireland", "Portugal"]
countries = ["Germany"]

#days = ["day1", "day2", "day3", "day4", "day5"]
days = ["day1"]


forecast_horizon = 96
training_size = 96 * 7 * 3 * 2
feature_selection = True
plot_forecast = True
hyperparameter_opt = True
opt_trials = 15

weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation"
]

# -------------------------
# Read JSON with forecast days
# -------------------------
with open(days_json_path, "r") as f:
    dataset_days = json.load(f)

# -------------------------
# Loop over countries
# -------------------------
for country in countries:
    print(f"\n{'#'*100}")
    print(f"COUNTRY: {country}")
    print(f"{'#'*100}")

    dataset_path = pathlib.Path(project_path) / "DataCleaning" / "clean" / f"dataset_{country}.csv"

    df_all = pd.read_csv(dataset_path, parse_dates=["timestamp"])
    df_all = df_all.set_index("timestamp")
    df_all = df_all.sort_index()

    home_cols = [col for col in df_all.columns if col.startswith("home_")]

    print(f"Detected {len(home_cols)} homes for {country}.")
    print(home_cols)
    # -------------------------
    # Loop over days
    # -------------------------
    for day_name in days:
        selected_day = dataset_days[country][day_name]
        date = f"{selected_day} 00:00:00"
        forecast_end_date = str(pd.Timestamp(date) + pd.Timedelta(days=1))

        print(f"\n{'='*100}")
        print(f"Running {country} - {day_name}")
        print(f"Forecast start: {date}")
        print(f"Forecast end:   {forecast_end_date}")
        print(f"{'='*100}")

        # -------------------------
        # Loop over homes
        # -------------------------
        for home_col in home_cols:
            print(f"\n{'-'*80}")
            print(f"Running forecast for {country} | {day_name} | {home_col}")
            print(f"{'-'*80}")

            try:
                forecasting_single_home(
                    project_path=project_path,
                    df_full=df_all[[home_col] + weather_cols].copy(),
                    target_col=home_col,
                    date=date,
                    forecast_end_date=forecast_end_date,
                    forecast_horizon=forecast_horizon,
                    training_size=training_size,
                    feature_selection=feature_selection,
                    plot_forecast=plot_forecast,
                    hyperparameter_opt=hyperparameter_opt,
                    models=models.copy(),
                    country=country,
                    day_name=day_name,
                    opt_trials=opt_trials
                )

                print(f"Finished {country} | {day_name} | {home_col}")

            except Exception as e:
                print(f"FAILED for {country} | {day_name} | {home_col}")
                print(f"Reason: {e}")
                continue


####################################################################################################
COUNTRY: Germany
####################################################################################################
Detected 28 homes for Germany.
['home_1', 'home_2', 'home_3', 'home_4', 'home_5', 'home_6', 'home_7', 'home_8', 'home_9', 'home_10', 'home_11', 'home_12', 'home_13', 'home_14', 'home_15', 'home_16', 'home_17', 'home_18', 'home_19', 'home_20', 'home_21', 'home_22', 'home_23', 'home_24', 'home_25', 'home_26', 'home_27', 'home_28']

Running Germany - day1
Forecast start: 2019-12-29 00:00:00
Forecast end:   2019-12-30 00:00:00

--------------------------------------------------------------------------------
Running forecast for Germany | day1 | home_1
--------------------------------------------------------------------------------

Home: home_1 | iteration 1/1
home_1 | X_train shape after feature selection: (3840, 97)


# end

283 mins for 1 run with TFT

11 mins for 1 with no TFT
